# 01 — MDM2 pIC50 Data Preparation

## Project Overview

This notebook prepares a curated experimental dataset for ligand-based quantitative structure–activity relationship (QSAR) modeling of small-molecule inhibitors of human MDM2.

MDM2 is a negative regulator of the tumor-suppressor protein p53. Inhibition of the MDM2–p53 interaction can restore p53 activity and represents an important strategy in anticancer drug discovery.

Experimental inhibitory activity measurements are retrieved from ChEMBL for the human MDM2 target:

- **Target name:** MDM2
- **Target ChEMBL ID:** CHEMBL5023
- **Organism:** Homo sapiens
- **Activity type:** IC50
- **Standard unit:** nM

The purpose of this notebook is to retrieve, curate, standardize, and prepare these experimental measurements for subsequent pIC50 QSAR modeling.

---

## Objectives

The main objectives are to:

1. Identify the human MDM2 target in ChEMBL.
2. Retrieve experimental MDM2 IC50 activity records.
3. Inspect missing values, measurement units, and activity relations.
4. Assess the quality of available IC50 measurements.
5. Remove incomplete, invalid, censored, and duplicate activity records.
6. Inspect the distribution and range of experimental IC50 values.
7. Convert IC50 measurements to continuous pIC50 values.
8. Aggregate repeated measurements for identical molecular structures.
9. Assign auxiliary activity classes for descriptive analysis.
10. Standardize molecular SMILES using RDKit.
11. Remove duplicate standardized structures.
12. Save the final curated QSAR dataset for downstream feature generation and modeling.

---

## Activity Endpoint

The primary experimental activity endpoint is **IC50**, reported in nanomolar (nM) units.

IC50 values are converted to pIC50 using:

$$
\mathrm{pIC50} = 9 - \log_{10}\left(\mathrm{IC50\ in\ nM}\right)
$$

Higher pIC50 values indicate greater experimentally measured inhibitory potency against MDM2.

The continuous pIC50 value is retained as the target variable for subsequent QSAR regression modeling.

## Auxiliary Activity Classes

For descriptive analysis, compounds are additionally assigned to three activity categories:

| Activity class | IC50 range |
|---|---:|
| Active | IC50 ≤ 1,000 nM |
| Intermediate | 1,000 nM < IC50 < 10,000 nM |
| Inactive | IC50 ≥ 10,000 nM |

These labels are used only to characterize the activity distribution of the curated dataset.

They are not used as the target of the primary pIC50 regression model.

---

## Data-Curation Workflow

The notebook follows the sequence:

**ChEMBL target selection**  
→ **IC50 activity retrieval**  
→ **missing-value and unit inspection**  
→ **measurement-quality assessment**  
→ **exact activity filtering**  
→ **IC50 range inspection**  
→ **pIC50 conversion**  
→ **repeated-measurement aggregation**  
→ **activity-class assignment**  
→ **molecular standardization**  
→ **duplicate-structure removal**  
→ **final dataset export**

---

## Output of This Notebook

The final output is a curated molecular dataset containing standardized structures and experimental pIC50 values suitable for downstream QSAR modeling.

The dataset produced here is used in:

- **Notebook 02 — PaDEL 1D/2D and Morgan Feature Generation**
- **Notebook 03 — ExtraTrees pIC50 Modeling and Evaluation**

Feature generation, scaffold-based model splitting, feature preprocessing, model training, and model evaluation are intentionally performed in the subsequent notebooks rather than in this data-preparation notebook.

### Step 1: Import libraries
New_client for downloading ChEMBL bioactivity data

In [5]:
# ============================================================
# STEP 1: Import libraries
# ============================================================

import numpy as np
import pandas as pd

from chembl_webresource_client.new_client import new_client

print("Libraries imported successfully.")

/Users/maryta/miniconda3/envs/mdm2_reinvent4/lib/python3.11/site-packages/chembl_webresource_client/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __version__ = __import__('pkg_resources').get_distribution('chembl_webresource_client').version


Libraries imported successfully.


### Step 2: Select the human MDM2 target

This step searches ChEMBL for MDM2 and selects the human MDM2 target:

In [6]:
# ============================================================
# STEP 2: Select the human MDM2 target
# ============================================================

target = new_client.target

target_query = target.search("MDM2")

targets = pd.DataFrame.from_dict(
    target_query
)

selected_target = targets.loc[
    (targets["target_chembl_id"] == "CHEMBL5023")
    & (targets["organism"] == "Homo sapiens"),
    "target_chembl_id",
].iloc[0]

print("Selected target:", selected_target)

Selected target: CHEMBL5023


### Step 3: Download MDM2 IC50 activity data
Explanation

This step retrieves compounds tested against human MDM2 with experimentally measured IC50 values.

It also downloads fields needed for later cleaning, including:

IC50 value
units
relation symbols such as =, <, or >
canonical SMILES
assay information

In [9]:
# ============================================================
# STEP 3: Download MDM2 IC50 activity data
# ============================================================

activity = new_client.activity

res = activity.filter(
    target_chembl_id=selected_target,
    standard_type="IC50",
).only([
    "molecule_chembl_id",
    "canonical_smiles",
    "standard_value",
    "standard_type",
    "standard_units",
    "standard_relation",
    "assay_chembl_id",
    "data_validity_comment",
    "potential_duplicate",
])

df_raw = pd.DataFrame.from_records(
    res
)

print("Raw activity records:", len(df_raw))

display(
    df_raw.head()
)

Raw activity records: 5821


,assay_chembl_id,canonical_smiles,data_validity_comment,molecule_chembl_id,potential_duplicate,relation,standard_relation,standard_type,standard_units,standard_value,type,units,value
0,CHEMBL874553,O=C(O)[C@H](c1ccccc1)N1C(=O)c2cc(I)ccc2NC(=O)[...,NaN,CHEMBL178578,0,=,=,IC50,nM,1700.0,IC50,uM,1.7
1,CHEMBL874553,O=C(O)[C@H](c1ccccc1)N1C(=O)c2cc(I)ccc2NC(=O)[...,NaN,CHEMBL178578,0,=,=,IC50,nM,1700.0,IC50,uM,1.7
2,CHEMBL874553,O=C(O)[C@H](c1ccc(Cl)cc1)N1C(=O)c2cc(I)ccc2NC(...,NaN,CHEMBL361103,0,=,=,IC50,nM,200.0,IC50,uM,0.2
3,CHEMBL874553,C[C@H](c1ccc(Cl)cc1)N(C(=O)/C=C/c1ccccc1)[C@H]...,NaN,CHEMBL182051,0,=,=,IC50,nM,30000.0,IC50,uM,30.0
4,CHEMBL874553,C[C@H](c1ccc(Cl)cc1)N1C(=O)C=C(c2ccccc2)N(CCCC...,NaN,CHEMBL360540,0,=,=,IC50,nM,13000.0,IC50,uM,13.0


### Step 4: Inspect missing values, units, and relations

Before cleaning, inspect whether there are:

missing SMILES
missing IC50 values
non-nM measurements
censored values such as < or >

In [10]:
# ============================================================
# STEP 4: Inspect raw IC50 data
# ============================================================

print("Dataset shape:", df_raw.shape)

print("\nMissing values")
print("-" * 50)
print(df_raw.isna().sum())

print("\nStandard units")
print("-" * 50)
print(
    df_raw["standard_units"]
    .value_counts(dropna=False)
)

print("\nStandard relations")
print("-" * 50)
print(
    df_raw["standard_relation"]
    .value_counts(dropna=False)
)

print("\nPotential duplicate values")
print("-" * 50)
print(
    df_raw["potential_duplicate"]
    .value_counts(dropna=False)
)

Dataset shape: (5821, 13)

Missing values
--------------------------------------------------
assay_chembl_id             0
canonical_smiles            4
data_validity_comment    5801
molecule_chembl_id          0
potential_duplicate         0
relation                   25
standard_relation          25
standard_type               0
standard_units             24
standard_value             24
type                        0
units                      24
value                      24
dtype: int64

Standard units
--------------------------------------------------
standard_units
nM         5796
NaN          24
ug.mL-1       1
Name: count, dtype: int64

Standard relations
--------------------------------------------------
standard_relation
=      5419
<       250
>       127
NaN      25
Name: count, dtype: int64

Potential duplicate values
--------------------------------------------------
potential_duplicate
0    5445
1     376
Name: count, dtype: int64


### Step 5: Inspect IC50 measurement quality

This step checks:

whether measurements are reported in nM
whether values are exact (=) or censored (<, >)
which records have validity warnings
how many records ChEMBL marks as potential duplicates

In [11]:
# ============================================================
# STEP 5: Inspect IC50 measurement quality
# ============================================================

print("\n" + "=" * 72)
print("STEP 5: Inspect IC50 measurement quality")
print("=" * 72)


print("\nStandard units")
print("-" * 50)
print(
    df_raw["standard_units"]
    .value_counts(dropna=False)
)


print("\nStandard relations")
print("-" * 50)
print(
    df_raw["standard_relation"]
    .value_counts(dropna=False)
)


print("\nData-validity comments")
print("-" * 50)
print(
    df_raw["data_validity_comment"]
    .value_counts(dropna=False)
)


print("\nPotential duplicate flags")
print("-" * 50)
print(
    df_raw["potential_duplicate"]
    .value_counts(dropna=False)
)


STEP 5: Inspect IC50 measurement quality

Standard units
--------------------------------------------------
standard_units
nM         5796
NaN          24
ug.mL-1       1
Name: count, dtype: int64

Standard relations
--------------------------------------------------
standard_relation
=      5419
<       250
>       127
NaN      25
Name: count, dtype: int64

Data-validity comments
--------------------------------------------------
data_validity_comment
NaN                              5801
Outside typical range              18
Potential transcription error       2
Name: count, dtype: int64

Potential duplicate flags
--------------------------------------------------
potential_duplicate
0    5445
1     376
Name: count, dtype: int64


### Step 6: Clean exact MDM2 IC50 records

In [12]:
# ============================================================
# STEP 6: Clean exact MDM2 IC50 records
# ============================================================

print("\n" + "=" * 72)
print("STEP 6: Clean exact MDM2 IC50 records")
print("=" * 72)

import pandas as pd


# ------------------------------------------------------------
# Start from a copy of the raw ChEMBL activity data
# ------------------------------------------------------------

df_clean = df_raw.copy()

initial_count = len(df_clean)


# ------------------------------------------------------------
# Convert IC50 values to numeric
# ------------------------------------------------------------

df_clean["standard_value"] = pd.to_numeric(
    df_clean["standard_value"],
    errors="coerce",
)


# ------------------------------------------------------------
# Remove rows missing essential fields
# ------------------------------------------------------------

df_clean = df_clean.dropna(
    subset=[
        "canonical_smiles",
        "standard_value",
        "standard_units",
        "standard_relation",
    ]
).copy()


# ------------------------------------------------------------
# Remove blank SMILES strings
# ------------------------------------------------------------

df_clean = df_clean[
    df_clean["canonical_smiles"]
    .astype(str)
    .str.strip()
    .ne("")
].copy()


# ------------------------------------------------------------
# Keep positive IC50 values only
# ------------------------------------------------------------

df_clean = df_clean[
    df_clean["standard_value"] > 0
].copy()


# ------------------------------------------------------------
# Keep nanomolar measurements only
# ------------------------------------------------------------

df_clean = df_clean[
    df_clean["standard_units"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("nm")
].copy()


# ------------------------------------------------------------
# Keep exact measurements only
# ------------------------------------------------------------
# Exclude censored values such as "<" and ">".

df_clean = df_clean[
    df_clean["standard_relation"]
    .astype(str)
    .str.strip()
    .eq("=")
].copy()


# ------------------------------------------------------------
# Remove records with ChEMBL validity warnings
# ------------------------------------------------------------

df_clean = df_clean[
    df_clean["data_validity_comment"].isna()
].copy()


# ------------------------------------------------------------
# Remove exact duplicated activity rows
# ------------------------------------------------------------

duplicate_subset = [
    "molecule_chembl_id",
    "canonical_smiles",
    "standard_value",
    "standard_units",
    "standard_relation",
    "assay_chembl_id",
]

exact_duplicate_count = df_clean.duplicated(
    subset=duplicate_subset
).sum()

df_clean = df_clean.drop_duplicates(
    subset=duplicate_subset
).copy()


# ------------------------------------------------------------
# Reset index
# ------------------------------------------------------------

df_clean = df_clean.reset_index(drop=True)


# ------------------------------------------------------------
# Print cleaning summary
# ------------------------------------------------------------

print("Initial records:", initial_count)
print("Exact duplicate rows removed:", exact_duplicate_count)
print("Clean records:", len(df_clean))
print("Total records removed:", initial_count - len(df_clean))

print("\nRemaining units")
print("-" * 50)
print(
    df_clean["standard_units"]
    .value_counts(dropna=False)
)

print("\nRemaining relations")
print("-" * 50)
print(
    df_clean["standard_relation"]
    .value_counts(dropna=False)
)

print("\nRemaining missing essential values")
print("-" * 50)
print(
    df_clean[
        [
            "canonical_smiles",
            "standard_value",
            "standard_units",
            "standard_relation",
        ]
    ].isna().sum()
)

display(df_clean.head())

print("\nStep 6 completed successfully.")


STEP 6: Clean exact MDM2 IC50 records
Initial records: 5821
Exact duplicate rows removed: 29
Clean records: 5373
Total records removed: 448

Remaining units
--------------------------------------------------
standard_units
nM    5373
Name: count, dtype: int64

Remaining relations
--------------------------------------------------
standard_relation
=    5373
Name: count, dtype: int64

Remaining missing essential values
--------------------------------------------------
canonical_smiles     0
standard_value       0
standard_units       0
standard_relation    0
dtype: int64


,assay_chembl_id,canonical_smiles,data_validity_comment,molecule_chembl_id,potential_duplicate,relation,standard_relation,standard_type,standard_units,standard_value,type,units,value
0,CHEMBL874553,O=C(O)[C@H](c1ccccc1)N1C(=O)c2cc(I)ccc2NC(=O)[...,NaN,CHEMBL178578,0,=,=,IC50,nM,1700.0,IC50,uM,1.7
1,CHEMBL874553,O=C(O)[C@H](c1ccc(Cl)cc1)N1C(=O)c2cc(I)ccc2NC(...,NaN,CHEMBL361103,0,=,=,IC50,nM,200.0,IC50,uM,0.2
2,CHEMBL874553,C[C@H](c1ccc(Cl)cc1)N(C(=O)/C=C/c1ccccc1)[C@H]...,NaN,CHEMBL182051,0,=,=,IC50,nM,30000.0,IC50,uM,30.0
3,CHEMBL874553,C[C@H](c1ccc(Cl)cc1)N1C(=O)C=C(c2ccccc2)N(CCCC...,NaN,CHEMBL360540,0,=,=,IC50,nM,13000.0,IC50,uM,13.0
4,CHEMBL874553,C[C@H](c1ccc(Cl)cc1)N1C(=O)C=C(c2ccccc2Br)N(CC...,NaN,CHEMBL427316,0,=,=,IC50,nM,3600.0,IC50,uM,3.6



Step 6 completed successfully.


### Step 6.1: Check and cap extreme IC50 values

This step checks whether any cleaned IC50 values are above:

100,000,000 nM

If there are extreme values, they are capped at that limit. The original standard_value column is kept, and a new column called standard_value_norm is created.

In [13]:
# ============================================================
# STEP 6.1: Check and cap extreme IC50 values
# ============================================================

print("\n" + "=" * 72)
print("STEP 6.1: Check and cap extreme IC50 values")
print("=" * 72)

IC50_MAX_NM = 100_000_000


# ------------------------------------------------------------
# Count extreme IC50 values
# ------------------------------------------------------------

extreme_count = (
    df_clean["standard_value"] > IC50_MAX_NM
).sum()

print(
    "Values above 100,000,000 nM:",
    extreme_count,
)

print(
    "Maximum original IC50 value:",
    df_clean["standard_value"].max(),
)


# ------------------------------------------------------------
# Create capped IC50 column
# ------------------------------------------------------------

df_clean["standard_value_norm"] = (
    df_clean["standard_value"]
    .clip(upper=IC50_MAX_NM)
)


# ------------------------------------------------------------
# Confirm the result
# ------------------------------------------------------------

print(
    "Maximum normalized IC50 value:",
    df_clean["standard_value_norm"].max(),
)

print(
    "Values capped:",
    (
        df_clean["standard_value"]
        != df_clean["standard_value_norm"]
    ).sum(),
)

display(
    df_clean[
        [
            "molecule_chembl_id",
            "standard_value",
            "standard_value_norm",
        ]
    ].sort_values(
        by="standard_value",
        ascending=False,
    ).head(10)
)

print("\nStep 6.1 completed successfully.")


STEP 6.1: Check and cap extreme IC50 values
Values above 100,000,000 nM: 0
Maximum original IC50 value: 100000.0
Maximum normalized IC50 value: 100000.0
Values capped: 0


,molecule_chembl_id,standard_value,standard_value_norm
5231,CHEMBL5926622,100000.0,100000.0
4560,CHEMBL5848493,98000.0,98000.0
1560,CHEMBL1688152,96000.0,96000.0
4551,CHEMBL5862449,93000.0,93000.0
4540,CHEMBL6026248,92000.0,92000.0
4536,CHEMBL5817293,92000.0,92000.0
65,CHEMBL426321,90000.0,90000.0
4559,CHEMBL5970325,90000.0,90000.0
1557,CHEMBL1688161,89000.0,89000.0
4561,CHEMBL5942474,89000.0,89000.0



Step 6.1 completed successfully.


### Step 7: Convert IC50 to pIC50


This step converts the cleaned IC50 values from nanomolar into pIC50.

The formula is:

pIC50 = 9 - log10(IC50 in nM)

Higher pIC50 means stronger activity.

In [14]:
# ============================================================
# STEP 7: Convert IC50 in nM to pIC50
# ============================================================

print("\n" + "=" * 72)
print("STEP 7: Convert IC50 to pIC50")
print("=" * 72)

import numpy as np


# ------------------------------------------------------------
# Convert IC50 in nM to pIC50
# Formula: pIC50 = 9 - log10(IC50_nM)
# ------------------------------------------------------------

df_clean["pIC50"] = (
    9.0
    - np.log10(df_clean["standard_value"])
)


# ------------------------------------------------------------
# Print summary statistics
# ------------------------------------------------------------

print("pIC50 summary")
print("-" * 50)

print(
    df_clean["pIC50"].describe()
)


# ------------------------------------------------------------
# Display example rows
# ------------------------------------------------------------

display(
    df_clean[
        [
            "molecule_chembl_id",
            "canonical_smiles",
            "standard_value",
            "standard_units",
            "pIC50",
        ]
    ].head()
)

print("\nStep 7 completed successfully.")


STEP 7: Convert IC50 to pIC50
pIC50 summary
--------------------------------------------------
count    5373.000000
mean        7.695872
std         1.317172
min         4.000000
25%         6.920819
50%         7.920819
75%         8.638272
max        10.397940
Name: pIC50, dtype: float64


,molecule_chembl_id,canonical_smiles,standard_value,standard_units,pIC50
0,CHEMBL178578,O=C(O)[C@H](c1ccccc1)N1C(=O)c2cc(I)ccc2NC(=O)[...,1700.0,nM,5.769551
1,CHEMBL361103,O=C(O)[C@H](c1ccc(Cl)cc1)N1C(=O)c2cc(I)ccc2NC(...,200.0,nM,6.698970
2,CHEMBL182051,C[C@H](c1ccc(Cl)cc1)N(C(=O)/C=C/c1ccccc1)[C@H]...,30000.0,nM,4.522879
3,CHEMBL360540,C[C@H](c1ccc(Cl)cc1)N1C(=O)C=C(c2ccccc2)N(CCCC...,13000.0,nM,4.886057
4,CHEMBL427316,C[C@H](c1ccc(Cl)cc1)N1C(=O)C=C(c2ccccc2Br)N(CC...,3600.0,nM,5.443697



Step 7 completed successfully.


### Step 8: Aggregate repeated measurements

In [15]:
# ============================================================
# STEP 8: Aggregate repeated measurements by SMILES
# ============================================================

print("\n" + "=" * 72)
print("STEP 8: Aggregate repeated measurements")
print("=" * 72)

df_pic50 = (
    df_clean
    .groupby(
        "canonical_smiles",
        as_index=False,
    )
    .agg(
        molecule_chembl_id=(
            "molecule_chembl_id",
            "first",
        ),
        pIC50=(
            "pIC50",
            "median",
        ),
        pIC50_mean=(
            "pIC50",
            "mean",
        ),
        pIC50_std=(
            "pIC50",
            "std",
        ),
        measurement_count=(
            "pIC50",
            "count",
        ),
        IC50_nM_median=(
            "standard_value",
            "median",
        ),
    )
)

print("Activity records before aggregation:", len(df_clean))
print("Unique SMILES after aggregation:", len(df_pic50))

print("\nMeasurement-count summary")
print("-" * 50)
print(df_pic50["measurement_count"].describe())

display(df_pic50.head())

print("\nStep 8 completed successfully.")


STEP 8: Aggregate repeated measurements
Activity records before aggregation: 5373
Unique SMILES after aggregation: 4146

Measurement-count summary
--------------------------------------------------
count    4146.000000
mean        1.295948
std         0.782787
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max        22.000000
Name: measurement_count, dtype: float64


,canonical_smiles,molecule_chembl_id,pIC50,pIC50_mean,pIC50_std,measurement_count,IC50_nM_median
0,C#CCC1CN(c2nc3nc(C(=O)O)nc(N[C@H](C)C4CCC4)c3n...,CHEMBL5986166,7.465466,7.465466,NaN,1,34.24
1,C#Cc1ccc(CN2C(=O)c3cc(C(C)(C)O)cc(F)c3[C@]2(OC...,CHEMBL5741921,8.116422,8.156187,0.085584,4,7.65
2,C#Cc1ccc(CN2C(=O)c3cc(C(C)(C)O)ccc3[C@]2(OCC2(...,CHEMBL4865855,7.958607,7.834080,0.215687,3,11.00
3,C#Cc1ccc(CN2C(=O)c3cc(C(C)(O)CO)cc(F)c3[C@]2(O...,CHEMBL5821902,8.283997,8.386497,0.319556,3,5.20
4,C#Cc1ccc(CN2C(=O)c3cc([C@@](C)(O)CO)cc(F)c3[C@...,CHEMBL5825373,8.283997,8.283997,NaN,1,5.20



Step 8 completed successfully.


### Step 9: Add activity classes for analysis

Active:        IC50 ≤ 1,000 nM
Intermediate:  1,000 < IC50 < 10,000 nM
Inactive:      IC50 ≥ 10,000 nM



In [16]:
# ============================================================
# STEP 9: Assign bioactivity classes
# ============================================================

print("\n" + "=" * 72)
print("STEP 9: Assign bioactivity classes")
print("=" * 72)


# ------------------------------------------------------------
# Define the activity classification function
# ------------------------------------------------------------

def classify_ic50(ic50_nm):
    """
    Classify a compound using its median IC50 in nM.
    """

    if ic50_nm <= 1_000:
        return "active"

    elif ic50_nm < 10_000:
        return "intermediate"

    else:
        return "inactive"


# ------------------------------------------------------------
# Apply the classification to each unique molecule
# ------------------------------------------------------------

df_pic50["bioactivity_class"] = (
    df_pic50["IC50_nM_median"]
    .apply(classify_ic50)
)


# ------------------------------------------------------------
# Print the class distribution
# ------------------------------------------------------------

print("Bioactivity class distribution")
print("-" * 50)

print(
    df_pic50["bioactivity_class"]
    .value_counts()
)

print("\nBioactivity class percentages")
print("-" * 50)

print(
    df_pic50["bioactivity_class"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)


# ------------------------------------------------------------
# Display example molecules
# ------------------------------------------------------------

display(
    df_pic50[
        [
            "molecule_chembl_id",
            "canonical_smiles",
            "IC50_nM_median",
            "pIC50",
            "bioactivity_class",
            "measurement_count",
        ]
    ].head(10)
)

print("\nStep 9 completed successfully.")


STEP 9: Assign bioactivity classes
Bioactivity class distribution
--------------------------------------------------
bioactivity_class
active          3633
intermediate     324
inactive         189
Name: count, dtype: int64

Bioactivity class percentages
--------------------------------------------------
bioactivity_class
active          87.63
intermediate     7.81
inactive         4.56
Name: proportion, dtype: float64


,molecule_chembl_id,canonical_smiles,IC50_nM_median,pIC50,bioactivity_class,measurement_count
0,CHEMBL5986166,C#CCC1CN(c2nc3nc(C(=O)O)nc(N[C@H](C)C4CCC4)c3n...,34.24,7.465466,active,1
1,CHEMBL5741921,C#Cc1ccc(CN2C(=O)c3cc(C(C)(C)O)cc(F)c3[C@]2(OC...,7.65,8.116422,active,4
2,CHEMBL4865855,C#Cc1ccc(CN2C(=O)c3cc(C(C)(C)O)ccc3[C@]2(OCC2(...,11.00,7.958607,active,3
3,CHEMBL5821902,C#Cc1ccc(CN2C(=O)c3cc(C(C)(O)CO)cc(F)c3[C@]2(O...,5.20,8.283997,active,3
4,CHEMBL5825373,C#Cc1ccc(CN2C(=O)c3cc([C@@](C)(O)CO)cc(F)c3[C@...,5.20,8.283997,active,1
5,CHEMBL380214,C#Cc1ccc2c(c1)C(=O)N([C@H](C)c1ccc(Cl)cc1)[C@@...,10300.00,4.987163,inactive,3
6,CHEMBL1689397,C=C(C)C(=O)NCCCC[C@H](NC(=O)[C@H](CCC(=O)O)NC(...,170.00,6.769551,active,1
7,CHEMBL1689403,C=C(C)C(=O)NCCCC[C@H](NC(=O)[C@H](Cc1ccccc1)NC...,14.00,7.853872,active,1
8,CHEMBL1689401,C=C(C)C(=O)NCCCC[C@H](NC(=O)[C@H](Cc1ccccc1)NC...,39.00,7.408935,active,1
9,CHEMBL1689402,C=C(C)C(=O)NCCCC[C@H](NC(=O)[C@H](Cc1ccccc1)NC...,28.00,7.552842,active,1



Step 9 completed successfully.


### Step 10: Save the cleaned QSAR dataset

This saves the final molecule-level dataset after:

cleaning IC50 values
converting IC50 to pIC50
combining repeated measurements using the median
assigning activity classes

In [17]:
# ============================================================
# STEP 10: Save the cleaned QSAR dataset
# ============================================================

print("\n" + "=" * 72)
print("STEP 10: Save the cleaned QSAR dataset")
print("=" * 72)

from pathlib import Path


# ------------------------------------------------------------
# Define output path
# ------------------------------------------------------------

OUTPUT_FILE = Path(
    "/Users/maryta/Downloads/mdm2_Reinvent_4/data/processed/"
    "mdm2_pic50_qsar.csv"
)

OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# Select final columns
# ------------------------------------------------------------

final_columns = [
    "molecule_chembl_id",
    "canonical_smiles",
    "IC50_nM_median",
    "pIC50",
    "pIC50_mean",
    "pIC50_std",
    "measurement_count",
    "bioactivity_class",
]

df_qsar = df_pic50[
    final_columns
].copy()


# ------------------------------------------------------------
# Save dataset
# ------------------------------------------------------------

df_qsar.to_csv(
    OUTPUT_FILE,
    index=False,
)


# ------------------------------------------------------------
# Confirm result
# ------------------------------------------------------------

print("Saved file:", OUTPUT_FILE)
print("Final dataset shape:", df_qsar.shape)

display(df_qsar.head())

print("\nStep 10 completed successfully.")


STEP 10: Save the cleaned QSAR dataset
Saved file: /Users/maryta/Downloads/mdm2_Reinvent_4/data/processed/mdm2_pic50_qsar.csv
Final dataset shape: (4146, 8)


,molecule_chembl_id,canonical_smiles,IC50_nM_median,pIC50,pIC50_mean,pIC50_std,measurement_count,bioactivity_class
0,CHEMBL5986166,C#CCC1CN(c2nc3nc(C(=O)O)nc(N[C@H](C)C4CCC4)c3n...,34.24,7.465466,7.465466,NaN,1,active
1,CHEMBL5741921,C#Cc1ccc(CN2C(=O)c3cc(C(C)(C)O)cc(F)c3[C@]2(OC...,7.65,8.116422,8.156187,0.085584,4,active
2,CHEMBL4865855,C#Cc1ccc(CN2C(=O)c3cc(C(C)(C)O)ccc3[C@]2(OCC2(...,11.00,7.958607,7.834080,0.215687,3,active
3,CHEMBL5821902,C#Cc1ccc(CN2C(=O)c3cc(C(C)(O)CO)cc(F)c3[C@]2(O...,5.20,8.283997,8.386497,0.319556,3,active
4,CHEMBL5825373,C#Cc1ccc(CN2C(=O)c3cc([C@@](C)(O)CO)cc(F)c3[C@...,5.20,8.283997,8.283997,NaN,1,active



Step 10 completed successfully.


### Step 11: Standardize and validate the SMILES


remove invalid SMILES
convert each molecule to a consistent canonical SMILES
avoid duplicate structures written in different ways

In [18]:
# ============================================================
# STEP 11: Standardize and validate SMILES
# ============================================================

print("\n" + "=" * 72)
print("STEP 11: Standardize and validate SMILES")
print("=" * 72)

from rdkit import Chem


def standardize_smiles(smiles):
    """
    Convert a SMILES string into a valid canonical SMILES.
    Returns None if the SMILES cannot be parsed.
    """

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    return Chem.MolToSmiles(
        mol,
        canonical=True,
        isomericSmiles=True,
    )


# Create standardized SMILES
df_qsar["standardized_smiles"] = (
    df_qsar["canonical_smiles"]
    .apply(standardize_smiles)
)


# Count invalid molecules
invalid_count = (
    df_qsar["standardized_smiles"]
    .isna()
    .sum()
)

print("Invalid SMILES:", invalid_count)


# Remove invalid molecules
df_qsar = (
    df_qsar
    .dropna(subset=["standardized_smiles"])
    .reset_index(drop=True)
)


print("Valid molecules remaining:", len(df_qsar))

display(
    df_qsar[
        [
            "molecule_chembl_id",
            "canonical_smiles",
            "standardized_smiles",
            "pIC50",
        ]
    ].head()
)

print("\nStep 11 completed successfully.")


STEP 11: Standardize and validate SMILES
Invalid SMILES: 0
Valid molecules remaining: 4146


,molecule_chembl_id,canonical_smiles,standardized_smiles,pIC50
0,CHEMBL5986166,C#CCC1CN(c2nc3nc(C(=O)O)nc(N[C@H](C)C4CCC4)c3n...,C#CCC1CN(c2nc3nc(C(=O)O)nc(N[C@H](C)C4CCC4)c3n...,7.465466
1,CHEMBL5741921,C#Cc1ccc(CN2C(=O)c3cc(C(C)(C)O)cc(F)c3[C@]2(OC...,C#Cc1ccc(CN2C(=O)c3cc(C(C)(C)O)cc(F)c3[C@]2(OC...,8.116422
2,CHEMBL4865855,C#Cc1ccc(CN2C(=O)c3cc(C(C)(C)O)ccc3[C@]2(OCC2(...,C#Cc1ccc(CN2C(=O)c3cc(C(C)(C)O)ccc3[C@]2(OCC2(...,7.958607
3,CHEMBL5821902,C#Cc1ccc(CN2C(=O)c3cc(C(C)(O)CO)cc(F)c3[C@]2(O...,C#Cc1ccc(CN2C(=O)c3cc(C(C)(O)CO)cc(F)c3[C@]2(O...,8.283997
4,CHEMBL5825373,C#Cc1ccc(CN2C(=O)c3cc([C@@](C)(O)CO)cc(F)c3[C@...,C#Cc1ccc(CN2C(=O)c3cc([C@@](C)(O)CO)cc(F)c3[C@...,8.283997



Step 11 completed successfully.


### Step 12: Remove duplicate standardized structures

Even after grouping by the original ChEMBL canonical_smiles, two rows can still become the same structure after RDKit standardization.

checks duplicate standardized_smiles
keeps one row per standardized molecule
reports how many duplicates were removed


In [19]:
# ============================================================
# STEP 12: Consolidate duplicate standardized structures
# ============================================================

print("\n" + "=" * 72)
print("STEP 12: Consolidate duplicate standardized structures")
print("=" * 72)


# ------------------------------------------------------------
# Count duplicate standardized SMILES
# ------------------------------------------------------------

duplicate_count = (
    df_qsar["standardized_smiles"]
    .duplicated()
    .sum()
)

print(
    "Duplicate standardized structures:",
    duplicate_count,
)


# ------------------------------------------------------------
# Combine rows that become the same standardized structure
# ------------------------------------------------------------

df_qsar = (
    df_qsar
    .groupby(
        "standardized_smiles",
        as_index=False,
    )
    .agg(
        molecule_chembl_id=(
            "molecule_chembl_id",
            "first",
        ),
        canonical_smiles=(
            "canonical_smiles",
            "first",
        ),
        IC50_nM_median=(
            "IC50_nM_median",
            "median",
        ),
        pIC50=(
            "pIC50",
            "median",
        ),
        pIC50_mean=(
            "pIC50",
            "mean",
        ),
        pIC50_std=(
            "pIC50",
            "std",
        ),
        measurement_count=(
            "measurement_count",
            "sum",
        ),
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Recreate activity classes
# ------------------------------------------------------------

df_qsar["bioactivity_class"] = (
    df_qsar["IC50_nM_median"]
    .apply(classify_ic50)
)


# ------------------------------------------------------------
# Confirm final molecule count
# ------------------------------------------------------------

print(
    "Unique standardized molecules remaining:",
    len(df_qsar),
)

print(
    "Remaining duplicate structures:",
    df_qsar["standardized_smiles"]
    .duplicated()
    .sum(),
)


display(
    df_qsar[
        [
            "molecule_chembl_id",
            "standardized_smiles",
            "pIC50",
            "measurement_count",
            "bioactivity_class",
        ]
    ].head()
)

print("\nStep 12 completed successfully.")


STEP 12: Consolidate duplicate standardized structures
Duplicate standardized structures: 0
Unique standardized molecules remaining: 4146
Remaining duplicate structures: 0


,molecule_chembl_id,standardized_smiles,pIC50,measurement_count,bioactivity_class
0,CHEMBL5986166,C#CCC1CN(c2nc3nc(C(=O)O)nc(N[C@H](C)C4CCC4)c3n...,7.465466,1,active
1,CHEMBL5741921,C#Cc1ccc(CN2C(=O)c3cc(C(C)(C)O)cc(F)c3[C@]2(OC...,8.116422,4,active
2,CHEMBL4865855,C#Cc1ccc(CN2C(=O)c3cc(C(C)(C)O)ccc3[C@]2(OCC2(...,7.958607,3,active
3,CHEMBL5821902,C#Cc1ccc(CN2C(=O)c3cc(C(C)(O)CO)cc(F)c3[C@]2(O...,8.283997,3,active
4,CHEMBL5825373,C#Cc1ccc(CN2C(=O)c3cc([C@@](C)(O)CO)cc(F)c3[C@...,8.283997,1,active



Step 12 completed successfully.


### Step 13: Save the final standardized QSAR dataset

In [20]:
# ============================================================
# STEP 13: Save the final standardized QSAR dataset
# ============================================================

print("\n" + "=" * 72)
print("STEP 13: Save the final standardized QSAR dataset")
print("=" * 72)

from pathlib import Path


# ------------------------------------------------------------
# Define the final output path
# ------------------------------------------------------------

FINAL_QSAR_FILE = Path(
    "/Users/maryta/Downloads/mdm2_Reinvent_4/data/processed/"
    "mdm2_pic50_qsar_standardized.csv"
)

FINAL_QSAR_FILE.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# Select final columns
# ------------------------------------------------------------

final_standardized_columns = [
    "molecule_chembl_id",
    "canonical_smiles",
    "standardized_smiles",
    "IC50_nM_median",
    "pIC50",
    "pIC50_mean",
    "pIC50_std",
    "measurement_count",
    "bioactivity_class",
]


# ------------------------------------------------------------
# Create the final saved DataFrame
# ------------------------------------------------------------

df_qsar_final = (
    df_qsar[
        final_standardized_columns
    ]
    .copy()
)


# ------------------------------------------------------------
# Final quality checks
# ------------------------------------------------------------

if df_qsar_final["standardized_smiles"].isna().any():
    raise ValueError(
        "The final dataset contains missing standardized SMILES."
    )

if df_qsar_final["standardized_smiles"].duplicated().any():
    raise ValueError(
        "The final dataset still contains duplicate structures."
    )

if df_qsar_final["pIC50"].isna().any():
    raise ValueError(
        "The final dataset contains missing pIC50 values."
    )


# ------------------------------------------------------------
# Save the final dataset
# ------------------------------------------------------------

df_qsar_final.to_csv(
    FINAL_QSAR_FILE,
    index=False,
)


# ------------------------------------------------------------
# Confirm the result
# ------------------------------------------------------------

print("Saved file:", FINAL_QSAR_FILE)
print("Final dataset shape:", df_qsar_final.shape)

print(
    "Duplicate standardized structures:",
    df_qsar_final["standardized_smiles"]
    .duplicated()
    .sum(),
)

print(
    "Missing pIC50 values:",
    df_qsar_final["pIC50"]
    .isna()
    .sum(),
)

display(
    df_qsar_final.head()
)

print("\nStep 13 completed successfully.")


STEP 13: Save the final standardized QSAR dataset
Saved file: /Users/maryta/Downloads/mdm2_Reinvent_4/data/processed/mdm2_pic50_qsar_standardized.csv
Final dataset shape: (4146, 9)
Duplicate standardized structures: 0
Missing pIC50 values: 0


,molecule_chembl_id,canonical_smiles,standardized_smiles,IC50_nM_median,pIC50,pIC50_mean,pIC50_std,measurement_count,bioactivity_class
0,CHEMBL5986166,C#CCC1CN(c2nc3nc(C(=O)O)nc(N[C@H](C)C4CCC4)c3n...,C#CCC1CN(c2nc3nc(C(=O)O)nc(N[C@H](C)C4CCC4)c3n...,34.24,7.465466,7.465466,NaN,1,active
1,CHEMBL5741921,C#Cc1ccc(CN2C(=O)c3cc(C(C)(C)O)cc(F)c3[C@]2(OC...,C#Cc1ccc(CN2C(=O)c3cc(C(C)(C)O)cc(F)c3[C@]2(OC...,7.65,8.116422,8.116422,NaN,4,active
2,CHEMBL4865855,C#Cc1ccc(CN2C(=O)c3cc(C(C)(C)O)ccc3[C@]2(OCC2(...,C#Cc1ccc(CN2C(=O)c3cc(C(C)(C)O)ccc3[C@]2(OCC2(...,11.00,7.958607,7.958607,NaN,3,active
3,CHEMBL5821902,C#Cc1ccc(CN2C(=O)c3cc(C(C)(O)CO)cc(F)c3[C@]2(O...,C#Cc1ccc(CN2C(=O)c3cc(C(C)(O)CO)cc(F)c3[C@]2(O...,5.20,8.283997,8.283997,NaN,3,active
4,CHEMBL5825373,C#Cc1ccc(CN2C(=O)c3cc([C@@](C)(O)CO)cc(F)c3[C@...,C#Cc1ccc(CN2C(=O)c3cc([C@@](C)(O)CO)cc(F)c3[C@...,5.20,8.283997,8.283997,NaN,1,active



Step 13 completed successfully.
